In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from itertools import permutations
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

# --- 1. CONFIGURATION & FILE MAPS ---
dataset_files = {
    'eICU': 'EICU_CARDIAC_ML_READY.csv',
    'MIMIC-3': 'MIMIC_III_CARDIAC_ML_READY.csv',
    'MIMIC-4': 'training_dataset.csv'
}

# --- AUTOMATIC COLUMN DETECTOR ---
def find_metadata_columns(df, dataset_name):
    """Dynamically finds the target (y) and patient group id columns based on common medical ML headers."""
    columns = df.columns.tolist()

    # Common names for the target variable (labels)
    possible_targets = ['y', 'label', 'mortality', 'hospitalmortality', 'actualhospitalmortality', 'target', 'outcome']
    # Common names for patient grouping
    possible_groups = ['hadm_id', 'patientunitstayid', 'stay_id', 'subject_id', 'icustay_id', 'encounter_id']

    target_col = next((c for c in columns if c.lower() in possible_targets), None)
    group_col = next((c for c in columns if c.lower() in possible_groups), None)

    if not target_col:
        # Fallback: check if 'y' or 'label' is a substring of any column
        target_col = next((c for c in columns if 'label' in c.lower() or 'mort' in c.lower()), None)
    if not group_col:
        target_col = next((c for c in columns if 'id' in c.lower()), None)

    if not target_col or not group_col:
        raise KeyError(f"Could not automatically find target or group ID in {dataset_name}. "
                       f"Columns found: {columns[:10]}...")

    print(f"   [{dataset_name}] Detected Target: '{target_col}' | Group ID: '{group_col}'")
    return target_col, group_col

# --- 2. PYTORCH MODEL DEFINITION ---
class LogisticRegressionPyTorch(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegressionPyTorch, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# --- 3. TRAINING & EVALUATION FUNCTION ---
def train_and_evaluate(train_name, test_name, df_train, df_test):
    print(f"\n" + "="*60)
    print(f" TRAINING: {train_name}  --->  TESTING: {test_name}")
    print("="*60)

    # Dynamically locate the target and group IDs for BOTH datasets
    train_target, train_group = find_metadata_columns(df_train, train_name)
    test_target, test_group = find_metadata_columns(df_test, test_name)

    # Intersect features: Find features present in BOTH datasets, excluding metadata
    train_ignore = [train_target, train_group]
    test_ignore = [test_target, test_group]

    feature_cols = [col for col in df_train.columns if col not in train_ignore and col in df_test.columns and col not in test_ignore]
    print(f" -> Training with {len(feature_cols)} aligned features.")

    X_train_raw = df_train[feature_cols].values
    y_train_raw = df_train[train_target].values

    X_test_raw = df_test[feature_cols].values
    y_test_raw = df_test[test_target].values
    test_groups = df_test[test_group].values

    # Handle missing values if any exist across mismatched datasets
    if np.isnan(X_train_raw).any() or np.isnan(X_test_raw).any():
        from sklearn.impute import SimpleImputer
        imputer = SimpleImputer(strategy='median')
        X_train_raw = imputer.fit_transform(X_train_raw)
        X_test_raw = imputer.transform(X_test_raw)

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    # Convert to PyTorch Tensors
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_raw, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    # Calculate Class Weights for Imbalance
    pos_weight = (len(y_train_raw) - sum(y_train_raw)) / sum(y_train_raw)
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32)

    # Data loader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

    # Initialize Model, Optimizer (Weight decay = L2 / Ridge Penalty)
    model = LogisticRegressionPyTorch(input_dim=len(feature_cols))
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-3)

    # Training Loop
    model.train()
    for epoch in range(15):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model.linear(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation Mode
    model.eval()
    with torch.no_grad():
        y_prob = model(X_test_tensor).numpy().flatten()

    # Build summary dataframe for Patient-Level Grouping
    test_summary = pd.DataFrame({
        'group_id': test_groups,
        'y_true': y_test_raw,
        'y_prob': y_prob
    })

    # Group by patient stay/admission ID and average the predictions
    patient_res = test_summary.groupby('group_id').mean()
    preds = (patient_res['y_prob'] >= 0.5).astype(int)

    # Metrics
    auc_score = roc_auc_score(patient_res['y_true'], patient_res['y_prob'])
    print(f"\n--- Patient-Level Evaluation [{train_name} -> {test_name}] ---")
    print(classification_report(patient_res['y_true'], preds))
    print(f"Patient-Level ROC-AUC Score: {auc_score:.4f}")

    return auc_score

# --- 4. EXECUTION LOOP ---
datasets = {}
for name, file_path in dataset_files.items():
    if os.path.exists(file_path):
        print(f"Loading {name} from {file_path}...")
        datasets[name] = pd.read_csv(file_path)
    else:
        print(f"⚠️ Warning: File {file_path} for {name} not found. Skipping.")

# Cross-validation combinations (Permutations of 2)
all_permutations = list(permutations(datasets.keys(), 2))
results_matrix = {}

for train_ds, test_ds in all_permutations:
    try:
        auc = train_and_evaluate(
            train_name=train_ds,
            test_name=test_ds,
            df_train=datasets[train_ds],
            df_test=datasets[test_ds]
        )
        results_matrix[f"{train_ds} -> {test_ds}"] = auc
    except Exception as e:
        print(f"❌ Failed processing permutation {train_ds} -> {test_ds}. Error: {e}")

# --- 5. FINAL SUMMARY ---
print("\n" + "="*60)
print("FINAL CROSS-DATASET PERFORMANCE SUMMARY")
print("="*60)
for experiment, score in results_matrix.items():
    print(f"{experiment:<25} : Patient ROC-AUC = {score:.4f}")

Loading eICU from EICU_CARDIAC_ML_READY.csv...
Loading MIMIC-3 from MIMIC_III_CARDIAC_ML_READY.csv...
Loading MIMIC-4 from training_dataset.csv...

 TRAINING: eICU  --->  TESTING: MIMIC-3
❌ Failed processing permutation eICU -> MIMIC-3. Error: "Could not automatically find target or group ID in eICU. Columns found: ['patientunitstayid', 'icd_code', 'icd_description', 'anchor_age', 'is_male', 'visit_procedure_count', 'vital_mean_heart_rate', 'vital_mean_systolic_bp', 'vital_mean_diastolic_bp', 'vital_mean_spo2']..."

 TRAINING: eICU  --->  TESTING: MIMIC-4
❌ Failed processing permutation eICU -> MIMIC-4. Error: "Could not automatically find target or group ID in eICU. Columns found: ['patientunitstayid', 'icd_code', 'icd_description', 'anchor_age', 'is_male', 'visit_procedure_count', 'vital_mean_heart_rate', 'vital_mean_systolic_bp', 'vital_mean_diastolic_bp', 'vital_mean_spo2']..."

 TRAINING: MIMIC-3  --->  TESTING: eICU
❌ Failed processing permutation MIMIC-3 -> eICU. Error: "Could n

In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from itertools import permutations
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# Set seed for reproducible PyTorch weights and splits
torch.manual_seed(42)
np.random.seed(42)

# --- 1. DATASET SPECIFIC COLUMN MAPS ---
# Explicitly matching structural columns based on your screenshots
dataset_configs = {
    'eICU': {
        'file': 'EICU_CARDIAC_ML_READY.csv',
        'group_col': 'patientunitstayid',
        'target_col': 'hospital_expire_flag'
    },
    'MIMIC-3': {
        'file': 'MIMIC_III_CARDIAC_ML_READY.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    },
    'MIMIC-4': {
        'file': 'training_dataset.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    }
}

# --- 2. PYTORCH MODEL ARCHITECTURE ---
class PyTorchRidgeLogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(PyTorchRidgeLogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# --- 3. TRAIN & INTER-DATASET EVALUATION FUNCTION ---
def run_cross_validation_pair(train_name, test_name, df_train, df_test):
    print(f"\n" + "="*70)
    print(f" EXPERIMENT: Train on [{train_name}]  ===>  Test on [{test_name}]")
    print("="*70)

    train_cfg = dataset_configs[train_name]
    test_cfg = dataset_configs[test_name]

    # 1. Identify valid numeric columns for each dataset, ignoring text descriptions
    text_cols = ['icd_code', 'icd_description', 'gender', 'subject_id']

    train_ignore = [train_cfg['target_col'], train_cfg['group_col']] + text_cols
    test_ignore = [test_cfg['target_col'], test_cfg['group_col']] + text_cols

    # Align features: dynamic intersection of what exists in BOTH dataframes
    feature_cols = [
        col for col in df_train.columns
        if col not in train_ignore and col in df_test.columns and col not in test_ignore
    ]

    print(f" -> Automatically aligned {len(feature_cols)} shared numerical features.")

    # 2. Extract arrays
    X_train_raw = df_train[feature_cols].values.astype(np.float32)
    y_train_raw = df_train[train_cfg['target_col']].values.astype(np.float32)

    X_test_raw = df_test[feature_cols].values.astype(np.float32)
    y_test_raw = df_test[test_cfg['target_col']].values.astype(np.float32)
    test_groups = df_test[test_cfg['group_col']].values

    # Impute missing values if data discrepancies exist across distinct systems
    if np.isnan(X_train_raw).any() or np.isnan(X_test_raw).any():
        from sklearn.impute import SimpleImputer
        imputer = SimpleImputer(strategy='median')
        X_train_raw = imputer.fit_transform(X_train_raw)
        X_test_raw = imputer.transform(X_test_raw)

    # 3. Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    # 4. Transform to PyTorch Tensors
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_raw, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    # 5. Handle imbalance using balancing pos_weight calculation
    neg_count = len(y_train_raw) - sum(y_train_raw)
    pos_weight_val = neg_count / max(1.0, sum(y_train_raw))
    pos_weight_tensor = torch.tensor([pos_weight_val], dtype=torch.float32)

    # Create Data Loader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

    # 6. Initialize Model & Optimization
    model = PyTorchRidgeLogisticRegression(input_dim=len(feature_cols))
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    # weight_decay adds L2 regularization, matching the 'Ridge' parameter in your original code
    optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)

    # Train the PyTorch model
    model.train()
    for epoch in range(25):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model.linear(batch_X)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

    # 7. Model Evaluation
    model.eval()
    with torch.no_grad():
        y_prob = model(X_test_tensor).numpy().flatten()

    # Build evaluation DataFrame back up to aggregate sequence predictions by unique patient
    test_summary = pd.DataFrame({
        'group_id': test_groups,
        'y_true': y_test_raw,
        'y_prob': y_prob
    })

    # Compute patient-level predictions (group tracking)
    patient_res = test_summary.groupby('group_id').mean()
    preds = (patient_res['y_prob'] >= 0.5).astype(int)

    # Metrics
    auc_score = roc_auc_score(patient_res['y_true'], patient_res['y_prob'])
    print(f"\n--- Patient-Level Evaluation Report [{train_name} -> {test_name}] ---")
    print(classification_report(patient_res['y_true'], preds, zero_division=0))
    print(f"ROC-AUC Score: {auc_score:.4f}")

    return auc_score

# --- 4. DATA LOADING ENGINE ---
datasets = {}
for name, config in dataset_configs.items():
    file_path = config['file']
    if os.path.exists(file_path):
        print(f"Successfully loaded {name} from workspace paths.")
        datasets[name] = pd.read_csv(file_path)
    else:
        print(f"❌ Error: File {file_path} for {name} missing from path!")

# Generate all 2-sequence combinations (Permutations)
# E.g., (eICU -> MIMIC-4), (MIMIC-4 -> MIMIC-3), (eICU -> MIMIC-3)...
cross_dataset_pairs = list(permutations(datasets.keys(), 2))
final_performance_matrix = {}

# Execute pipeline sequentially across combinations
for train_dataset_name, test_dataset_name in cross_dataset_pairs:
    try:
        auc = run_cross_validation_pair(
            train_name=train_dataset_name,
            test_name=test_dataset_name,
            df_train=datasets[train_dataset_name],
            df_test=datasets[test_dataset_name]
        )
        final_performance_matrix[f"{train_dataset_name} -> {test_dataset_name}"] = auc
    except Exception as e:
        print(f"❌ Could not execute configuration {train_dataset_name} -> {test_dataset_name}. Details: {e}")

# --- 5. SUMMARY PRINT OUT ---
print("\n" + "="*70)
print("             ALL-PAIR CROSS DATASET EVALUATION MATRIX")
print("="*70)
for trajectory, score in final_performance_matrix.items():
    print(f"{trajectory:<28} | Patient-Level ROC-AUC = {score:.4f}")

Successfully loaded eICU from workspace paths.
Successfully loaded MIMIC-3 from workspace paths.
Successfully loaded MIMIC-4 from workspace paths.

 EXPERIMENT: Train on [eICU]  ===>  Test on [MIMIC-3]
 -> Automatically aligned 8 shared numerical features.

--- Patient-Level Evaluation Report [eICU -> MIMIC-3] ---
              precision    recall  f1-score   support

         0.0       0.92      0.68      0.79     31451
         1.0       0.24      0.65      0.35      4945

    accuracy                           0.68     36396
   macro avg       0.58      0.67      0.57     36396
weighted avg       0.83      0.68      0.73     36396

ROC-AUC Score: 0.7284

 EXPERIMENT: Train on [eICU]  ===>  Test on [MIMIC-4]
 -> Automatically aligned 8 shared numerical features.

--- Patient-Level Evaluation Report [eICU -> MIMIC-4] ---
              precision    recall  f1-score   support

         0.0       0.99      0.86      0.92      3366
         1.0       0.12      0.67      0.20        93

  

In [ ]:
# 2 and 1 at a time

In [6]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from itertools import combinations
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

# --- 1. CONFIGURATION & FILE MAPS ---
dataset_configs = {
    'eICU': {
        'file': 'EICU_CARDIAC_ML_READY.csv',
        'group_col': 'patientunitstayid',
        'target_col': 'hospital_expire_flag'
    },
    'MIMIC-3': {
        'file': 'MIMIC_III_CARDIAC_ML_READY.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    },
    'MIMIC-4': {
        'file': 'training_dataset.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    }
}

# --- 2. MODEL DEFINITION ---
class PyTorchLogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(PyTorchLogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# --- 3. CORE PROCESSING ENGINE ---
def execute_fusion_experiment(train_list, test_list, datasets):
    """
    Dynamically aligns features, combines multiple datasets for training/testing,
    trains a PyTorch model, and returns the patient-level evaluation.
    """
    print(f"\n" + "="*75)
    print(f" EXPERIMENT: Train on {train_list}  ===>  Test on {test_list}")
    print("="*75)

    text_cols = ['icd_code', 'icd_description', 'gender', 'subject_id']

    # Identify common feature columns across ALL datasets involved in this run
    all_involved = train_list + test_list
    shared_features = None

    for ds_name in all_involved:
        cfg = dataset_configs[ds_name]
        ignore = [cfg['target_col'], cfg['group_col']] + text_cols
        valid_features = [c for c in datasets[ds_name].columns if c not in ignore]

        if shared_features is None:
            shared_features = set(valid_features)
        else:
            shared_features = shared_features.intersection(set(valid_features))

    feature_cols = list(shared_features)
    print(f" -> Aligned {len(feature_cols)} shared numerical features across systems.")

    # Build Combined Training Set
    train_x_parts, train_y_parts = [], []
    for ds_name in train_list:
        cfg = dataset_configs[ds_name]
        df = datasets[ds_name]
        train_x_parts.append(df[feature_cols].values.astype(np.float32))
        train_y_parts.append(df[cfg['target_col']].values.astype(np.float32))

    X_train_raw = np.vstack(train_x_parts)
    y_train_raw = np.concatenate(train_y_parts)

    # Build Combined Testing Set (keeping track of group columns natively)
    test_x_parts, test_y_parts, test_group_parts = [], [], []
    for ds_name in test_list:
        cfg = dataset_configs[ds_name]
        df = datasets[ds_name]
        test_x_parts.append(df[feature_cols].values.astype(np.float32))
        test_y_parts.append(df[cfg['target_col']].values.astype(np.float32))
        # Unique mapping: prepend dataset name to prevent cross-dataset group collisions
        test_group_parts.append(ds_name + "_" + df[cfg['group_col']].astype(str))

    X_test_raw = np.vstack(test_x_parts)
    y_test_raw = np.concatenate(test_y_parts)
    test_groups = np.concatenate(test_group_parts)

    # Impute missing entries if any exist
    if np.isnan(X_train_raw).any() or np.isnan(X_test_raw).any():
        from sklearn.impute import SimpleImputer
        imputer = SimpleImputer(strategy='median')
        X_train_raw = imputer.fit_transform(X_train_raw)
        X_test_raw = imputer.transform(X_test_raw)

    # Scale variables
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    # Convert to Tensors
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_raw, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    # Class imbalance handling
    neg_count = len(y_train_raw) - sum(y_train_raw)
    pos_weight = neg_count / max(1.0, sum(y_train_raw))
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=512, shuffle=True)

    # Initialize & Optimize PyTorch module
    model = PyTorchLogisticRegression(input_dim=len(feature_cols))
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)

    model.train()
    for epoch in range(25):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model.linear(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        y_prob = model(X_test_tensor).numpy().flatten()

    test_summary = pd.DataFrame({
        'group_id': test_groups,
        'y_true': y_test_raw,
        'y_prob': y_prob
    })

    patient_res = test_summary.groupby('group_id').mean()
    preds = (patient_res['y_prob'] >= 0.5).astype(int)

    auc_score = roc_auc_score(patient_res['y_true'], patient_res['y_prob'])
    print(f"\n--- Combined Patient-Level Evaluation Report ---")
    print(classification_report(patient_res['y_true'], preds, zero_division=0))
    print(f"ROC-AUC Score: {auc_score:.4f}")

    return auc_score

# --- 4. DATA LOADING ---
datasets = {}
for name, config in dataset_configs.items():
    if os.path.exists(config['file']):
        print(f"Loaded {name} dataset successfully.")
        datasets[name] = pd.read_csv(config['file'])
    else:
        print(f"❌ Target file missing: {config['file']}")

all_keys = list(datasets.keys())
results_summary = {}

# --- 5. SEQUENTIAL EVALUATION LOOPS ---

# Trajectory A: 2 Datasets for Training -> 1 Dataset for Testing
print("\n" + "#"*50 + "\n RUNNING CONFIGURATION: 2 TRAIN -> 1 TEST \n" + "#"*50)
for train_pair in combinations(all_keys, 2):
    train_list = list(train_pair)
    test_list = [name for name in all_keys if name not in train_list]

    exp_name = f"Train: {('+').join(train_list)} -> Test: {test_list[0]}"
    try:
        auc = execute_fusion_experiment(train_list, test_list, datasets)
        results_summary[exp_name] = auc
    except Exception as e:
        print(f"❌ Failed combo {exp_name}. Error: {e}")

# Trajectory B: 1 Dataset for Training -> 2 Datasets for Testing
print("\n" + "#"*50 + "\n RUNNING CONFIGURATION: 1 TRAIN -> 2 TEST \n" + "#"*50)
for single_train in all_keys:
    train_list = [single_train]
    test_list = [name for name in all_keys if name != single_train]

    exp_name = f"Train: {single_train} -> Test: {('+').join(test_list)}"
    try:
        auc = execute_fusion_experiment(train_list, test_list, datasets)
        results_summary[exp_name] = auc
    except Exception as e:
        print(f"❌ Failed combo {exp_name}. Error: {e}")

# --- 6. PRINT COMBINED METRIC SUMMARY MATRIX ---
print("\n" + "="*75)
print("                  COMPLETE FUSION EVALUATION SUMMARY")
print("="*75)
for strategy, score in results_summary.items():
    print(f"{strategy:<50} | Unified Patient ROC-AUC = {score:.4f}")

Loaded eICU dataset successfully.
Loaded MIMIC-3 dataset successfully.
Loaded MIMIC-4 dataset successfully.

##################################################
 RUNNING CONFIGURATION: 2 TRAIN -> 1 TEST 
##################################################

 EXPERIMENT: Train on ['eICU', 'MIMIC-3']  ===>  Test on ['MIMIC-4']
 -> Aligned 8 shared numerical features across systems.

--- Combined Patient-Level Evaluation Report ---
              precision    recall  f1-score   support

         0.0       0.99      0.73      0.84      3366
         1.0       0.07      0.78      0.13        93

    accuracy                           0.73      3459
   macro avg       0.53      0.76      0.49      3459
weighted avg       0.97      0.73      0.82      3459

ROC-AUC Score: 0.8208

 EXPERIMENT: Train on ['eICU', 'MIMIC-4']  ===>  Test on ['MIMIC-3']
 -> Aligned 8 shared numerical features across systems.

--- Combined Patient-Level Evaluation Report ---
              precision    recall  f1-score  

In [ ]:
# all three down

In [7]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score

# Ensure absolute reproducibility
torch.manual_seed(42)
np.random.seed(42)

# --- 1. CONFIGURATION & FILE MAPS ---
dataset_configs = {
    'eICU': {
        'file': 'EICU_CARDIAC_ML_READY.csv',
        'group_col': 'patientunitstayid',
        'target_col': 'hospital_expire_flag'
    },
    'MIMIC-3': {
        'file': 'MIMIC_III_CARDIAC_ML_READY.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    },
    'MIMIC-4': {
        'file': 'training_dataset.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    }
}

# --- 2. MODEL DEFINITION ---
class PyTorchLogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(PyTorchLogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# --- 3. LOADING & ALIGNING ALL DATASETS ---
datasets = {}
for name, config in dataset_configs.items():
    if os.path.exists(config['file']):
        print(f"Loading {name} from {config['file']}...")
        datasets[name] = pd.read_csv(config['file'])
    else:
        print(f"❌ Target file missing: {config['file']}")

print("\n--- Aligning shared features across all 3 databases ---")
text_cols = ['icd_code', 'icd_description', 'gender', 'subject_id']
shared_features = None

for name in datasets.keys():
    cfg = dataset_configs[name]
    ignore = [cfg['target_col'], cfg['group_col']] + text_cols
    valid_features = [c for c in datasets[name].columns if c not in ignore]

    if shared_features is None:
        shared_features = set(valid_features)
    else:
        shared_features = shared_features.intersection(set(valid_features))

feature_cols = list(shared_features)
print(f"Aligned {len(feature_cols)} mutually shared numerical features.")

# --- 4. BUILDING THE MASTER COMBINED POOL ---
x_master_list = []
y_master_list = []
group_master_list = []

for name, df in datasets.items():
    cfg = dataset_configs[name]

    # Extract matrices
    x_master_list.append(df[feature_cols].values.astype(np.float32))
    y_master_list.append(df[cfg['target_col']].values.astype(np.float32))

    # Prevent ID collision by modifying the group keys safely
    unique_groups = name + "_" + df[cfg['group_col']].astype(str)
    group_master_list.append(unique_groups.values)

X_all = np.vstack(x_master_list)
y_all = np.concatenate(y_master_list)
groups_all = np.concatenate(group_master_list)

# Check for missing data across the combined matrix
if np.isnan(X_all).any():
    from sklearn.impute import SimpleImputer
    X_all = SimpleImputer(strategy='median').fit_transform(X_all)

print(f"\n✅ Master Pool Created Successfully!")
print(f"Total Combined Matrix Rows: {X_all.shape[0]} | Total Features: {X_all.shape[1]}")

# --- 5. ROBUST STRATIFIED CROSS-VALIDATION ENGINE ---
# Using 5-Fold Stratified CV to robustly validate the 3-in-1 unified data pool
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

print("\n" + "="*70)
print(" STARTING CROSS-VALIDATION ON ALL THREE COMBINED DATASETS")
print("="*70)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
    print(f"\nProcessing Fold {fold + 1}/5...")

    # Split into structural folds
    X_train_raw, X_test_raw = X_all[train_idx], X_all[test_idx]
    y_train_raw, y_test_raw = y_all[train_idx], y_all[test_idx]
    test_groups = groups_all[test_idx]

    # Standardize data locally to avoid validation lookahead leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    # Pack into PyTorch tensors
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_raw, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    # Compute balanced loss weight handling
    neg_count = len(y_train_raw) - sum(y_train_raw)
    pos_weight = neg_count / max(1.0, sum(y_train_raw))
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=512, shuffle=True)

    # Initialize Core Network & Optimizer (L2 Ridge weight decay penalty applied)
    model = PyTorchLogisticRegression(input_dim=X_all.shape[1])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)

    # Network Training Loop
    model.train()
    for epoch in range(20):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model.linear(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation Pass
    model.eval()
    with torch.no_grad():
        y_prob = model(X_test_tensor).numpy().flatten()

    # Aggregate predictions up to unique patient-admissions level
    fold_summary = pd.DataFrame({
        'group_id': test_groups,
        'y_true': y_test_raw,
        'y_prob': y_prob
    })

    patient_res = fold_summary.groupby('group_id').mean()
    preds = (patient_res['y_prob'] >= 0.5).astype(int)

    # Metrics calculation per fold
    auc_score = roc_auc_score(patient_res['y_true'], patient_res['y_prob'])
    cv_scores.append(auc_score)
    print(f"Fold {fold + 1} Patient-Level ROC-AUC Score: {auc_score:.4f}")

# --- 6. FINAL RECAP REPORT ---
print("\n" + "="*70)
print("             ALL-THREE COMBINED DATA PERFORMANCE SUMMARY")
print("="*70)
print(f"Mean Patient-Level ROC-AUC Across Folds : {np.mean(cv_scores):.4f} (± {np.std(cv_scores):.4f})")
print(f"Individual Fold Array Scores           : {[round(x, 4) for x in cv_scores]}")

Loading eICU from EICU_CARDIAC_ML_READY.csv...
Loading MIMIC-3 from MIMIC_III_CARDIAC_ML_READY.csv...
Loading MIMIC-4 from training_dataset.csv...

--- Aligning shared features across all 3 databases ---
Aligned 8 mutually shared numerical features.

✅ Master Pool Created Successfully!
Total Combined Matrix Rows: 147851 | Total Features: 8

 STARTING CROSS-VALIDATION ON ALL THREE COMBINED DATASETS

Processing Fold 1/5...
Fold 1 Patient-Level ROC-AUC Score: 0.7451

Processing Fold 2/5...
Fold 2 Patient-Level ROC-AUC Score: 0.7464

Processing Fold 3/5...
Fold 3 Patient-Level ROC-AUC Score: 0.7449

Processing Fold 4/5...
Fold 4 Patient-Level ROC-AUC Score: 0.7521

Processing Fold 5/5...
Fold 5 Patient-Level ROC-AUC Score: 0.7480

             ALL-THREE COMBINED DATA PERFORMANCE SUMMARY
Mean Patient-Level ROC-AUC Across Folds : 0.7473 (± 0.0026)
Individual Fold Array Scores           : [np.float64(0.7451), np.float64(0.7464), np.float64(0.7449), np.float64(0.7521), np.float64(0.748)]


In [ ]:
# Combined MIMIC Pool Cross-Validation

In [8]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score

# Ensure absolute reproducibility
torch.manual_seed(42)
np.random.seed(42)

# --- 1. MIMIC-ONLY CONFIGURATION ---
mimic_configs = {
    'MIMIC-3': {
        'file': 'MIMIC_III_CARDIAC_ML_READY.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    },
    'MIMIC-4': {
        'file': 'training_dataset.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    }
}

# --- 2. PYTORCH MODEL ---
class PyTorchLogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(PyTorchLogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# --- 3. LOADING & ALIGNING MIMIC FEATURES ---
datasets = {}
for name, config in mimic_configs.items():
    if os.path.exists(config['file']):
        print(f"Loading {name} from {config['file']}...")
        datasets[name] = pd.read_csv(config['file'])
    else:
        print(f"❌ Target file missing: {config['file']}! Make sure it is in your sidebar.")

print("\n--- Aligning shared features between MIMIC-3 and MIMIC-4 ---")
text_cols = ['icd_code', 'icd_description', 'gender', 'subject_id']
shared_features = None

for name in datasets.keys():
    cfg = mimic_configs[name]
    ignore = [cfg['target_col'], cfg['group_col']] + text_cols
    valid_features = [c for c in datasets[name].columns if c not in ignore]

    if shared_features is None:
        shared_features = set(valid_features)
    else:
        shared_features = shared_features.intersection(set(valid_features))

feature_cols = list(shared_features)
print(f"Aligned {len(feature_cols)} identical numerical features between both MIMIC systems.")

# --- 4. CONCATENATING INTO THE MASTER MIMIC POOL ---
x_parts = []
y_parts = []
group_parts = []

for name, df in datasets.items():
    cfg = mimic_configs[name]

    x_parts.append(df[feature_cols].values.astype(np.float32))
    y_parts.append(df[cfg['target_col']].values.astype(np.float32))

    # Prepend dataset name to avoid cross-version admission ID collisions
    unique_groups = name + "_" + df[cfg['group_col']].astype(str)
    group_parts.append(unique_groups.values)

X_mimic = np.vstack(x_parts)
y_mimic = np.concatenate(y_parts)
groups_mimic = np.concatenate(group_parts)

# Handle any missing values in the combined matrix safely
if np.isnan(X_mimic).any():
    from sklearn.impute import SimpleImputer
    X_mimic = SimpleImputer(strategy='median').fit_transform(X_mimic)

print(f"\n✅ Master MIMIC Pool Built!")
print(f"Total Combined MIMIC Rows: {X_mimic.shape[0]} | Continuous Features: {X_mimic.shape[1]}")

# --- 5. STRATIFIED CROSS-VALIDATION LOOP ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

print("\n" + "="*70)
print(" STARTING 5-FOLD CROSS-VALIDATION ON COMBINED MIMIC DATA POOL")
print("="*70)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_mimic, y_mimic)):
    print(f"\nProcessing Fold {fold + 1}/5...")

    # Stratified Splits
    X_train_raw, X_test_raw = X_mimic[train_idx], X_mimic[test_idx]
    y_train_raw, y_test_raw = y_mimic[train_idx], y_mimic[test_idx]
    test_groups = groups_mimic[test_idx]

    # Scale within the fold to avoid leaking information
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    # Convert to PyTorch Tensors
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_raw, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    # Imbalance adjustment (like class_weight='balanced')
    neg_count = len(y_train_raw) - sum(y_train_raw)
    pos_weight = neg_count / max(1.0, sum(y_train_raw))
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=512, shuffle=True)

    # Init PyTorch Layer with L2 Regularization (weight_decay)
    model = PyTorchLogisticRegression(input_dim=X_mimic.shape[1])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)

    # Train
    model.train()
    for epoch in range(20):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model.linear(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    # Test Evaluation Mode
    model.eval()
    with torch.no_grad():
        y_prob = model(X_test_tensor).numpy().flatten()

    # Group predictions back to the patient level
    fold_summary = pd.DataFrame({
        'group_id': test_groups,
        'y_true': y_test_raw,
        'y_prob': y_prob
    })

    patient_res = fold_summary.groupby('group_id').mean()
    preds = (patient_res['y_prob'] >= 0.5).astype(int)

    # Track Metrics
    auc_score = roc_auc_score(patient_res['y_true'], patient_res['y_prob'])
    cv_scores.append(auc_score)
    print(f"Fold {fold + 1} Patient-Level ROC-AUC Score: {auc_score:.4f}")

# --- 6. FINAL METRICS RECAP ---
print("\n" + "="*70)
print("             UNIFIED MIMIC POOL PERFORMANCE SUMMARY")
print("="*70)
print(f"Mean Patient-Level ROC-AUC Across Folds : {np.mean(cv_scores):.4f} (± {np.std(cv_scores):.4f})")
print(f"All Individual Fold Scores             : {[round(x, 4) for x in cv_scores]}")

Loading MIMIC-3 from MIMIC_III_CARDIAC_ML_READY.csv...
Loading MIMIC-4 from training_dataset.csv...

--- Aligning shared features between MIMIC-3 and MIMIC-4 ---
Aligned 13 identical numerical features between both MIMIC systems.

✅ Master MIMIC Pool Built!
Total Combined MIMIC Rows: 54151 | Continuous Features: 13

 STARTING 5-FOLD CROSS-VALIDATION ON COMBINED MIMIC DATA POOL

Processing Fold 1/5...
Fold 1 Patient-Level ROC-AUC Score: 0.7863

Processing Fold 2/5...
Fold 2 Patient-Level ROC-AUC Score: 0.7967

Processing Fold 3/5...
Fold 3 Patient-Level ROC-AUC Score: 0.8073

Processing Fold 4/5...
Fold 4 Patient-Level ROC-AUC Score: 0.7966

Processing Fold 5/5...
Fold 5 Patient-Level ROC-AUC Score: 0.8041

             UNIFIED MIMIC POOL PERFORMANCE SUMMARY
Mean Patient-Level ROC-AUC Across Folds : 0.7982 (± 0.0073)
All Individual Fold Scores             : [np.float64(0.7863), np.float64(0.7967), np.float64(0.8073), np.float64(0.7966), np.float64(0.8041)]


In [ ]:
# mimic full feature


In [12]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score

# Ensure absolute reproducibility across your runtime environment
torch.manual_seed(42)
np.random.seed(42)

# --- 1. MIMIC CONFIGURATION ---
mimic_configs = {
    'MIMIC-3': {
        'file': 'MIMIC_III_CARDIAC_ML_READY.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    },
    'MIMIC-4': {
        'file': 'training_dataset.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    }
}

# --- 2. LOADING DATASETS ---
datasets = {}
for name, config in mimic_configs.items():
    if os.path.exists(config['file']):
        print(f"Loading {name} from {config['file']}...")
        datasets[name] = pd.read_csv(config['file'])
    else:
        print(f"❌ Missing critical file: {config['file']}. Please check your sidebar paths.")

# --- 3. THE ANTI-LEAKAGE FILTER ENGINE ---
print("\n--- Constructing Maximum Feature Space (Filtering Out Target Proxies) ---")

# Standard textual metadata variables to drop
metadata_to_drop = ['icd_code', 'icd_description', 'gender', 'subject_id']

# Core leakage blacklist: features that act as a proxy mirror for death or retrospective text
leaked_features = [
    'is_deceased',
    'death',
    'discharge',
    'report_discharge',
    'hospital_expire_flag',
    'expire',
    'decompensation'
]

# Maximize features by computing the union of all valid available columns
maximized_features = set()
for name, df in datasets.items():
    cfg = mimic_configs[name]

    # Structural ignore pool: targets, keys, standard descriptions, and the leakage list
    ignore_list = [cfg['target_col'], cfg['group_col']] + metadata_to_drop + leaked_features
    valid_cols = [c for c in df.columns if c not in ignore_list]
    maximized_features.update(valid_cols)

feature_cols = list(maximized_features)
print(f"✅ Safely retained {len(feature_cols)} features (NLP report terms & numerical columns fully preserved).")

# --- 4. POOLING DATASETS WITH SAFE ZERO-PADDING ---
x_parts = []
y_parts = []
group_parts = []

for name, df in datasets.items():
    cfg = mimic_configs[name]

    # Reindex aligns feature dimensions perfectly across MIMIC iterations.
    # If an NLP feature exists in MIMIC-4 but not MIMIC-3, it is safely padded with 0.0.
    df_aligned = df.reindex(columns=feature_cols, fill_value=0.0)

    x_parts.append(df_aligned.values.astype(np.float32))
    y_parts.append(df[cfg['target_col']].values.astype(np.float32))

    # Prepend source tag to prevent inter-database admission tracking token collisions
    unique_groups = name + "_" + df[cfg['group_col']].astype(str)
    group_parts.append(unique_groups.values)

X_mimic = np.vstack(x_parts)
y_mimic = np.concatenate(y_parts)
groups_mimic = np.concatenate(group_parts)

# Secondary safety layer to handle loose NaNs across historical frames
if np.isnan(X_mimic).any():
    from sklearn.impute import SimpleImputer
    X_mimic = SimpleImputer(strategy='median').fit_transform(X_mimic)

print(f"\n✅ Balanced Master MIMIC Pool successfully established!")
print(f"Matrix Dimension: {X_mimic.shape[0]} rows × {X_mimic.shape[1]} features")

# --- 5. PYTORCH NEURAL LINEAR BASE ---
class PyTorchLogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(PyTorchLogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# --- 6. STRATIFIED CROSS-VALIDATION LOOP ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

print("\n" + "="*80)
print(" STARTING LEAKAGE-FREE CROSS-VALIDATION ON COMBINED MIMIC-3 & MIMIC-4 POOL")
print("="*80)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_mimic, y_mimic)):
    print(f"\nProcessing Fold {fold + 1}/5...")

    # Partition indices
    X_train_raw, X_test_raw = X_mimic[train_idx], X_mimic[test_idx]
    y_train_raw, y_test_raw = y_mimic[train_idx], y_mimic[test_idx]
    test_groups = groups_mimic[test_idx]

    # Local feature scaling inside each fold loop to prevent data leakage from lookahead averages
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    # Convert data matrices to PyTorch arrays
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_raw, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    # Compute inverse class balancing scale for stable binary loss calculation
    neg_count = len(y_train_raw) - sum(y_train_raw)
    pos_weight = neg_count / max(1.0, sum(y_train_raw))
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=512, shuffle=True)

    # Instantiate neural network architecture
    model = PyTorchLogisticRegression(input_dim=X_mimic.shape[1])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)

    # Train iteration blocks
    model.train()
    for epoch in range(20):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model.linear(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation pass over hidden test partition folds
    model.eval()
    with torch.no_grad():
        y_prob = model(X_test_tensor).numpy().flatten()

    # Collate raw row predictions back to standard patient-admission levels
    fold_summary = pd.DataFrame({
        'group_id': test_groups,
        'y_true': y_test_raw,
        'y_prob': y_prob
    })

    patient_res = fold_summary.groupby('group_id').mean()
    preds = (patient_res['y_prob'] >= 0.5).astype(int)

    # Measure evaluation scores safely
    auc_score = roc_auc_score(patient_res['y_true'], patient_res['y_prob'])
    cv_scores.append(auc_score)
    print(f"Fold {fold + 1} Patient-Level ROC-AUC Score: {auc_score:.4f}")

# --- 7. FINAL REPORT MATRIX ---
print("\n" + "="*80)
print("             CLEAN MULTI-VERSION MIMIC MATRIC ANALYSIS COMPLETE")
print("="*80)
print(f"Mean Patient-Level ROC-AUC Across Folds : {np.mean(cv_scores):.4f} (± {np.std(cv_scores):.4f})")
print(f"All Individual Fold Scores             : {[round(float(x), 4) for x in cv_scores]}")

Loading MIMIC-3 from MIMIC_III_CARDIAC_ML_READY.csv...
Loading MIMIC-4 from training_dataset.csv...

--- Constructing Maximum Feature Space (Filtering Out Target Proxies) ---
✅ Safely retained 162 features (NLP report terms & numerical columns fully preserved).

✅ Balanced Master MIMIC Pool successfully established!
Matrix Dimension: 54151 rows × 162 features

 STARTING LEAKAGE-FREE CROSS-VALIDATION ON COMBINED MIMIC-3 & MIMIC-4 POOL

Processing Fold 1/5...
Fold 1 Patient-Level ROC-AUC Score: 0.9549

Processing Fold 2/5...
Fold 2 Patient-Level ROC-AUC Score: 0.9540

Processing Fold 3/5...
Fold 3 Patient-Level ROC-AUC Score: 0.9545

Processing Fold 4/5...
Fold 4 Patient-Level ROC-AUC Score: 0.9534

Processing Fold 5/5...
Fold 5 Patient-Level ROC-AUC Score: 0.9582

             CLEAN MULTI-VERSION MIMIC MATRIC ANALYSIS COMPLETE
Mean Patient-Level ROC-AUC Across Folds : 0.9550 (± 0.0017)
All Individual Fold Scores             : [0.9549, 0.954, 0.9545, 0.9534, 0.9582]


In [10]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score

# Ensure absolute reproducibility
torch.manual_seed(42)
np.random.seed(42)

# --- 1. CONFIGURATION ---
dataset_configs = {
    'eICU': {
        'file': 'EICU_CARDIAC_ML_READY.csv',
        'group_col': 'patientunitstayid',
        'target_col': 'hospital_expire_flag'
    },
    'MIMIC-3': {
        'file': 'MIMIC_III_CARDIAC_ML_READY.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    },
    'MIMIC-4': {
        'file': 'training_dataset.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    }
}

# --- 2. LOADING & FIXING eICU COLUMNS FIRST ---
datasets = {}
for name, config in dataset_configs.items():
    if os.path.exists(config['file']):
        print(f"Loading {name} from {config['file']}...")
        df = pd.read_csv(config['file'])

        # FIXING THE eICU TRUNCATION BOTTLENECK
        if name == 'eICU':
            # Check for partial names and rename them to match MIMIC perfectly
            rename_map = {}
            for col in df.columns:
                if 'heart' in col.lower():
                    rename_map[col] = 'vital_mean_heart_rate'
                elif 'systolic' in col.lower():
                    rename_map[col] = 'vital_mean_systolic_bp'
                elif 'diastolic' in col.lower():
                    rename_map[col] = 'vital_mean_diastolic_bp'
                elif 'spo2' in col.lower():
                    rename_map[col] = 'vital_mean_spo2'
                elif 'temp' in col.lower():
                    rename_map[col] = 'vital_mean_temp'

            if rename_map:
                print(f"   -> Fixed {len(rename_map)} truncated eICU vital headers: {rename_map}")
                df = df.rename(columns=rename_map)

        datasets[name] = df
    else:
        print(f"❌ Target file missing: {config['file']}")

# --- 3. COMPILING THE GLOBAL UNION OF FEATURES ---
print("\n--- Compiling Global Feature Pool (Preserving all NLP and Vitals) ---")
metadata_to_drop = ['icd_code', 'icd_description', 'gender', 'subject_id']

global_features = set()
for name, df in datasets.items():
    cfg = dataset_configs[name]
    ignore_list = [cfg['target_col'], cfg['group_col']] + metadata_to_drop
    valid_cols = [c for c in df.columns if c not in ignore_list]
    global_features.update(valid_cols)

feature_cols = list(global_features)
print(f"Total Unified Feature space size: {len(feature_cols)} (NLP columns + aligned Vitals preserved)")

# --- 4. ALIGNING AND POOLING ALL THREE DATASETS ---
x_master = []
y_master = []
group_master = []

for name, df in datasets.items():
    cfg = dataset_configs[name]

    # Reindex pads eICU with 0.0 for text features, while perfectly retaining its vitals!
    df_aligned = df.reindex(columns=feature_cols, fill_value=0.0)

    x_master.append(df_aligned.values.astype(np.float32))
    y_master.append(df[cfg['target_col']].values.astype(np.float32))

    unique_groups = name + "_" + df[cfg['group_col']].astype(str)
    group_master.append(unique_groups.values)

X_all = np.vstack(x_master)
y_all = np.concatenate(y_master)
groups_all = np.concatenate(group_master)

# Dynamic Imputation for any hidden NaNs
if np.isnan(X_all).any():
    from sklearn.impute import SimpleImputer
    X_all = SimpleImputer(strategy='median').fit_transform(X_all)

print(f"\n✅ Upgraded Master Pool Complete!")
print(f"Matrix Dimension: {X_all.shape[0]} samples × {X_all.shape[1]} features")

# --- 5. STRATIFIED CROSS-VALIDATION ENGINE ---
class PyTorchLogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(PyTorchLogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

print("\n" + "="*75)
print(" RUNNING 5-FOLD CROSS-VALIDATION ON ULTIMATE HIGH-FIDELITY POOL")
print("="*75)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
    print(f"\nProcessing Fold {fold + 1}/5...")

    X_train_raw, X_test_raw = X_all[train_idx], X_all[test_idx]
    y_train_raw, y_test_raw = y_all[train_idx], y_all[test_idx]
    test_groups = groups_all[test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_raw, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    # Class balance weighting
    neg_count = len(y_train_raw) - sum(y_train_raw)
    pos_weight = neg_count / max(1.0, sum(y_train_raw))
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=512, shuffle=True)

    model = PyTorchLogisticRegression(input_dim=X_all.shape[1])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)

    model.train()
    for epoch in range(20):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model.linear(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        y_prob = model(X_test_tensor).numpy().flatten()

    fold_summary = pd.DataFrame({
        'group_id': test_groups,
        'y_true': y_test_raw,
        'y_prob': y_prob
    })

    patient_res = fold_summary.groupby('group_id').mean()
    preds = (patient_res['y_prob'] >= 0.5).astype(int)

    auc_score = roc_auc_score(patient_res['y_true'], patient_res['y_prob'])
    cv_scores.append(auc_score)
    print(f"Fold {fold + 1} Patient-Level ROC-AUC: {auc_score:.4f}")

# --- 6. FINAL PRESENTATION ---
print("\n" + "="*75)
print("             MAXIMIZED 3-DATASET POOL PERFORMANCE MATRIX")
print("="*75)
print(f"Mean Patient-Level ROC-AUC Across Folds : {np.mean(cv_scores):.4f} (± {np.std(cv_scores):.4f})")
print(f"All Individual Fold Scores             : {[round(x, 4) for x in cv_scores]}")

Loading eICU from EICU_CARDIAC_ML_READY.csv...
   -> Fixed 5 truncated eICU vital headers: {'vital_mean_heart_rate': 'vital_mean_heart_rate', 'vital_mean_systolic_bp': 'vital_mean_systolic_bp', 'vital_mean_diastolic_bp': 'vital_mean_diastolic_bp', 'vital_mean_spo2': 'vital_mean_spo2', 'vital_mean_temp': 'vital_mean_temp'}
Loading MIMIC-3 from MIMIC_III_CARDIAC_ML_READY.csv...
Loading MIMIC-4 from training_dataset.csv...

--- Compiling Global Feature Pool (Preserving all NLP and Vitals) ---
Total Unified Feature space size: 164 (NLP columns + aligned Vitals preserved)

✅ Upgraded Master Pool Complete!
Matrix Dimension: 147851 samples × 164 features

 RUNNING 5-FOLD CROSS-VALIDATION ON ULTIMATE HIGH-FIDELITY POOL

Processing Fold 1/5...
Fold 1 Patient-Level ROC-AUC: 0.8718

Processing Fold 2/5...
Fold 2 Patient-Level ROC-AUC: 0.8765

Processing Fold 3/5...
Fold 3 Patient-Level ROC-AUC: 0.8766

Processing Fold 4/5...
Fold 4 Patient-Level ROC-AUC: 0.8831

Processing Fold 5/5...
Fold 5 Pati

In [11]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score

# Ensure absolute reproducibility
torch.manual_seed(42)
np.random.seed(42)

# --- 1. CONFIGURATION ---
dataset_configs = {
    'eICU': {
        'file': 'EICU_CARDIAC_ML_READY.csv',
        'group_col': 'patientunitstayid',
        'target_col': 'hospital_expire_flag'
    },
    'MIMIC-3': {
        'file': 'MIMIC_III_CARDIAC_ML_READY.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    },
    'MIMIC-4': {
        'file': 'training_dataset.csv',
        'group_col': 'hadm_id',
        'target_col': 'hospital_expire_flag'
    }
}

# --- 2. LOADING & ALIGNING eICU COLUMNS ---
datasets = {}
for name, config in dataset_configs.items():
    if os.path.exists(config['file']):
        print(f"Loading {name} from {config['file']}...")
        df = pd.read_csv(config['file'])

        # FIXING THE eICU TRUNCATION BOTTLENECK
        if name == 'eICU':
            rename_map = {}
            for col in df.columns:
                if 'heart' in col.lower():
                    rename_map[col] = 'vital_mean_heart_rate'
                elif 'systolic' in col.lower():
                    rename_map[col] = 'vital_mean_systolic_bp'
                elif 'diastolic' in col.lower():
                    rename_map[col] = 'vital_mean_diastolic_bp'
                elif 'spo2' in col.lower():
                    rename_map[col] = 'vital_mean_spo2'
                elif 'temp' in col.lower():
                    rename_map[col] = 'vital_mean_temp'

            if rename_map:
                print(f"   -> Fixed {len(rename_map)} truncated eICU vital headers: {rename_map}")
                df = df.rename(columns=rename_map)

        datasets[name] = df
    else:
        print(f"❌ Target file missing from file sidebar: {config['file']}")

# --- 3. RIGOROUS ANTI-LEAKAGE FEATURE COMPILATION ---
print("\n--- Compiling Clean Feature Pool (Filtering Out Target Proxies) ---")

# Text/ID columns to drop completely
metadata_to_drop = ['icd_code', 'icd_description', 'gender', 'subject_id']

# Core leakage blacklist: features that trick the model by acting as a target mirror
leaked_features = [
    'is_deceased',
    'death',
    'discharge',
    'report_discharge',
    'hospital_expire_flag',
    'expire',
    'decompensation'
]

global_features = set()
for name, df in datasets.items():
    cfg = dataset_configs[name]

    # Ignore targets, group tracking keys, standard strings, and leaked columns
    ignore_list = [cfg['target_col'], cfg['group_col']] + metadata_to_drop + leaked_features
    valid_cols = [c for c in df.columns if c not in ignore_list]
    global_features.update(valid_cols)

feature_cols = list(global_features)
print(f"Total ethically retained features: {len(feature_cols)} (NLP columns + aligned Vitals preserved)")

# --- 4. DATASET UNION CONCATENATION ---
x_master = []
y_master = []
group_master = []

for name, df in datasets.items():
    cfg = dataset_configs[name]

    # Use reindex to cleanly introduce full NLP dimension blocks to eICU as zeros safely
    df_aligned = df.reindex(columns=feature_cols, fill_value=0.0)

    x_master.append(df_aligned.values.astype(np.float32))
    y_master.append(df[cfg['target_col']].values.astype(np.float32))

    # Prepend dataset identifier string to maintain absolute group uniqueness
    unique_groups = name + "_" + df[cfg['group_col']].astype(str)
    group_master.append(unique_groups.values)

X_all = np.vstack(x_master)
y_all = np.concatenate(y_master)
groups_all = np.concatenate(group_master)

# Clear hidden NaN fragments across databases
if np.isnan(X_all).any():
    from sklearn.impute import SimpleImputer
    X_all = SimpleImputer(strategy='median').fit_transform(X_all)

print(f"\n✅ Upgraded Master Data Pool Implemented Successfully!")
print(f"Clean Matrix Shape: {X_all.shape[0]} rows × {X_all.shape[1]} features")

# --- 5. PYTORCH LINEAR CLASSIFIER MODULE ---
class PyTorchLogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(PyTorchLogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# --- 6. STRATIFIED VALIDATION TRACKING ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

print("\n" + "="*80)
print(" RUNNING 5-FOLD CROSS-VALIDATION ON GROUND-TRUTH MEDICAL FEATURE POOL")
print("="*80)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
    print(f"\nProcessing Fold {fold + 1}/5...")

    # Split folds cleanly
    X_train_raw, X_test_raw = X_all[train_idx], X_all[test_idx]
    y_train_raw, y_test_raw = y_all[train_idx], y_all[test_idx]
    test_groups = groups_all[test_idx]

    # Apply standard scaling inside folds to completely safeguard against lookahead bias
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    # Format directly to PyTorch architecture
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_raw, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    # Dynamically compute class imbalance weights based on minority mortalities
    neg_count = len(y_train_raw) - sum(y_train_raw)
    pos_weight = neg_count / max(1.0, sum(y_train_raw))
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=512, shuffle=True)

    # Model instantiation
    model = PyTorchLogisticRegression(input_dim=X_all.shape[1])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    # Adam Optimization with steady L2 Ridge penalty weight decay config
    optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)

    # Execute Training Routine
    model.train()
    for epoch in range(20):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model.linear(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    # Inference Optimization pass
    model.eval()
    with torch.no_grad():
        y_prob = model(X_test_tensor).numpy().flatten()

    # Collate predictions up to true unique tracking identifiers
    fold_summary = pd.DataFrame({
        'group_id': test_groups,
        'y_true': y_test_raw,
        'y_prob': y_prob
    })

    patient_res = fold_summary.groupby('group_id').mean()
    preds = (patient_res['y_prob'] >= 0.5).astype(int)

    # Calculate performance metrics safely without proxy distortions
    auc_score = roc_auc_score(patient_res['y_true'], patient_res['y_prob'])
    cv_scores.append(auc_score)
    print(f"Fold {fold + 1} Patient-Level ROC-AUC Score: {auc_score:.4f}")

# --- 7. RECAP EVALUATION LOG ---
print("\n" + "="*80)
print("             MAXIMIZED REALISTIC PERFORMANCE MATRIX MATRIX")
print("="*80)
print(f"Mean Patient-Level ROC-AUC Across Folds : {np.mean(cv_scores):.4f} (± {np.std(cv_scores):.4f})")
print(f"All Individual Fold Scores             : {[round(float(x), 4) for x in cv_scores]}")

Loading eICU from EICU_CARDIAC_ML_READY.csv...
   -> Fixed 5 truncated eICU vital headers: {'vital_mean_heart_rate': 'vital_mean_heart_rate', 'vital_mean_systolic_bp': 'vital_mean_systolic_bp', 'vital_mean_diastolic_bp': 'vital_mean_diastolic_bp', 'vital_mean_spo2': 'vital_mean_spo2', 'vital_mean_temp': 'vital_mean_temp'}
Loading MIMIC-3 from MIMIC_III_CARDIAC_ML_READY.csv...
Loading MIMIC-4 from training_dataset.csv...

--- Compiling Clean Feature Pool (Filtering Out Target Proxies) ---
Total ethically retained features: 162 (NLP columns + aligned Vitals preserved)

✅ Upgraded Master Data Pool Implemented Successfully!
Clean Matrix Shape: 147851 rows × 162 features

 RUNNING 5-FOLD CROSS-VALIDATION ON GROUND-TRUTH MEDICAL FEATURE POOL

Processing Fold 1/5...
Fold 1 Patient-Level ROC-AUC Score: 0.8326

Processing Fold 2/5...
Fold 2 Patient-Level ROC-AUC Score: 0.8385

Processing Fold 3/5...
Fold 3 Patient-Level ROC-AUC Score: 0.8372

Processing Fold 4/5...
Fold 4 Patient-Level ROC-AUC 